In [1]:
import os
from dotenv import load_dotenv
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index
from rag_helper import RAGBase
from openai import OpenAI
from toyaikit.tools import Tools
from toyaikit.llm import OpenAIChatCompletionsClient
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIChatCompletionsRunner, DisplayingRunnerCallback

load_dotenv()

True

In [2]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
documents = []
for file in files:
    doc = file.parse()
    documents.append(doc)

In [4]:
index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)
index.fit(documents)

In [5]:
mistral_client = OpenAI(
    api_key=os.getenv("MISTRAL_API_KEY"),
    base_url="https://api.mistral.ai/v1"
)

In [6]:
question = "How does the agentic loop keep calling the model until it stops?"

In [7]:
assistant = RAGBase(
    index=index,
    llm_client=mistral_client
)

In [10]:
response = assistant.rag(question)
response

('The agentic loop keeps calling the model until it stops by using a `while` loop that continues as long as the model returns a function call (tool usage). Here\'s how it works based on the context:\n\n1. The loop starts by sending the current conversation history (messages) to the model.\n2. The model may respond with a function call (e.g., to use the `search` tool) or a final answer.\n3. If the model returns a function call:\n   - The code executes the tool (e.g., `search`)\n   - The tool\'s output is appended to the conversation history\n   - The loop continues (another iteration starts)\n4. If the model returns a final answer (no function calls):\n   - The loop breaks and the final answer is returned\n\nThe exit condition is simple: the loop stops when `has_function_calls` is `False` (meaning the model\'s latest response contained no tool calls).\n\nThis is implemented in the core loop:\n```python\nwhile True:\n    # ... send messages to model ...\n    has_function_calls = False\n\

In [11]:
chunks = chunk_documents(documents, size=2000, step=1000)
print(len(chunks))

295


In [12]:
index.fit(chunks)

In [13]:
def search(query: str) -> dict[str, str]:
    """
    Search the chunks for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
    )

In [14]:
instructions = """
    You're a course teaching assistant. 
Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
""".strip()

In [15]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [ ]:
llm_client = OpenAIChatCompletionsClient(
    model="mistral-small-latest",
    client=mistral_client
)

In [ ]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIChatCompletionsRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client
)

In [19]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received


Feature,Plain RAG,Agentic Loop
Workflow,"Fixed, linear pipeline","Dynamic, iterative process"
Adaptability,"Static, no adaptation",Adapts based on intermediate results
Tool Usage,No tool interactions,"Can interact with tools (e.g., APIs)"
Retrieval,Single-pass retrieval,Multi-turn retrieval and refinement
Error Handling,No self-correction,Can detect and correct errors
Use Case,Simple Q&A from a fixed knowledge base,Complex tasks requiring planning and tool usage
